# Regression Improvement — Outlier Removal + XGBoost + Neural Network

Starting point: Random Forest R² ≈ 0.22 on raw data.

This notebook tests three levers for improving regression metrics:
1. **Outlier removal** — find the right cut-off that reduces noise without losing too much data
2. **XGBoost** — stronger gradient boosting with built-in regularisation
3. **Neural network** — MLP regressor that can learn non-linear interactions

Each is tested on the same outlier-cleaned dataset so results are comparable.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.preprocessing  import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import (
    RandomForestRegressor, GradientBoostingRegressor
)
from sklearn.neural_network  import MLPRegressor
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline        import Pipeline

# XGBoost — optional
try:
    from xgboost import XGBRegressor
    XGBOOST_OK = True
    print('XGBoost available.')
except ImportError:
    XGBOOST_OK = False
    print('XGBoost not found — install with: pip install xgboost')

# TensorFlow / Keras — optional
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_OK = True
    print(f'TensorFlow {tf.__version__} available.')
except ImportError:
    TF_OK = False
    print('TensorFlow not found — install with: pip install tensorflow')

plt.rcParams.update({'figure.dpi': 130,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
C_BLUE, C_TEAL, C_RED, C_AMBER, C_GRAY = (
    '#378ADD', '#1D9E75', '#E24B4A', '#BA7517', '#888780'
)

DATA_PATH      = 'sampled_data.csv'
RANDOM_STATE   = 42
MIN_DEPT_COUNT = 500   # higher threshold for large datasets
TOP_N_SRT      = 20    # top service request types to keep

# ── Memory control ────────────────────────────────────────────────────
# For large files (>200k rows) set this below 1.0 to work on a
# stratified subsample. 0.2 = 20% of rows, stratified by Department.
# Set to 1.0 to use the full file.
SAMPLE_FRAC = 1.0

# ── Target transform ──────────────────────────────────────────────────
# 'raw'  — no transform (best R², worst MAE)
# 'sqrt' — square root (good balance — recommended)
# 'log'  — log1p (best MAE, lower raw R² but best log-scale R²)
TARGET_TRANSFORM = 'sqrt'

print('\nImports OK.')

In [ ]:
# ── Load & preprocess ────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded {len(df):,} rows')

# Stratified subsample by Department (for large files)
if SAMPLE_FRAC < 1.0:
    _counts = df['Department'].value_counts()
    _common = _counts[_counts >= MIN_DEPT_COUNT].index
    df['_s']  = df['Department'].where(df['Department'].isin(_common), 'Other')
    df = (df.groupby('_s', group_keys=False)
            .sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
            .drop(columns=['_s'])
            .reset_index(drop=True))
    print(f'Working sample ({SAMPLE_FRAC*100:.0f}%): {len(df):,} rows')

DATE_FMT = '%Y %b %d %I:%M:%S %p'
df['Created Date'] = pd.to_datetime(df['Created Date'], format=DATE_FMT, errors='coerce')
df['Closed Date']  = pd.to_datetime(df['Closed Date'],  format=DATE_FMT, errors='coerce')
df['hours_to_close'] = (
    (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600
)
# Remove exact zeros (auto-closures) and negatives
df = df[df['hours_to_close'] > 0].copy()

df['ERT_days']    = df['ERT (Estimated Response Time)'].str.extract(r'(\d+)')[0].astype(float)
df['month']       = df['Created Date'].dt.month
df['day_of_week'] = df['Created Date'].dt.dayofweek
df['hour']        = df['Created Date'].dt.hour
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

dept_counts  = df['Department'].value_counts()
common_depts = dept_counts[dept_counts >= MIN_DEPT_COUNT].index
df['Department_grouped'] = df['Department'].where(
    df['Department'].isin(common_depts), 'Other'
)

DROP_COLS = [
    'Service Request Number', 'Unique Key', 'Address', 'Department',
    'Closed Date', 'Update Date', 'Status', 'Outcome', 'Lat_Long Location',
    'Overall Service Request Due Date', 'Created Date',
    'ERT (Estimated Response Time)',
]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

ccd = pd.to_numeric(
    df['City Council District'].astype(str).str.split(',').str[0].str.strip(),
    errors='coerce'
)
df['City Council District'] = ccd.fillna(ccd.median())

top_types = df['Service Request Type'].value_counts().nlargest(TOP_N_SRT).index
df['Service Request Type'] = df['Service Request Type'].where(
    df['Service Request Type'].isin(top_types), 'Other'
)

ORD_COLS = [c for c in
    ['Priority', 'Method Received Description', 'Department_grouped']
    if c in df.columns]
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[ORD_COLS] = oe.fit_transform(df[ORD_COLS].astype(str))

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
srt_enc = ohe.fit_transform(df[['Service Request Type']])
df_srt  = pd.DataFrame(srt_enc, columns=ohe.get_feature_names_out(), index=df.index)
df = pd.concat([df.drop(columns=['Service Request Type']), df_srt], axis=1)

num_cols = df.select_dtypes(include='number').columns.difference(['hours_to_close'])
imp = SimpleImputer(strategy='median')
df[num_cols] = imp.fit_transform(df[num_cols])

print(f'Preprocessed: {len(df):,} rows × {df.shape[1]} columns')
print(f'Target — mean: {df["hours_to_close"].mean():.1f}h  '
      f'median: {df["hours_to_close"].median():.1f}h  '
      f'skew: {df["hours_to_close"].skew():.2f}')

## Section 1 — Outlier Removal Sweep

Testing different cut-off strategies to find the best balance between
removing noise and retaining data.

In [ ]:
# ── Outlier removal sweep ─────────────────────────────────────────────────
y_all = df['hours_to_close']
q75, q25 = y_all.quantile(0.75), y_all.quantile(0.25)
iqr = q75 - q25

STRATEGIES = {
    'No removal':        df,
    'Remove >99th pct':  df[y_all <= y_all.quantile(0.99)],
    'Remove >95th pct':  df[y_all <= y_all.quantile(0.95)],
    'Remove >90th pct':  df[y_all <= y_all.quantile(0.90)],
    'IQR ×1.5':          df[y_all <= q75 + 1.5 * iqr],
    'IQR ×3.0':          df[y_all <= q75 + 3.0 * iqr],
    'Cap at 720h (30d)': df[y_all <= 720],
}

rf_sweep = RandomForestRegressor(
    n_estimators=100, max_depth=10, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1
)

print('Running outlier sweep (5-fold CV)...\n')
print(f'{"Strategy":<22} {"Rows":>8} {"Removed":>9} {"Skew":>7} {"CV R²":>9} {"CV RMSE":>9}')
print('-' * 70)

sweep_results = []
for name, d in STRATEGIES.items():
    y = d['hours_to_close']
    X = d.drop(columns=['hours_to_close'])
    removed = len(df) - len(d)
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    r2_scores   = cross_val_score(rf_sweep, X, y, cv=cv, scoring='r2', n_jobs=-1)
    rmse_scores = np.sqrt(-cross_val_score(
        rf_sweep, X, y, cv=cv,
        scoring='neg_mean_squared_error', n_jobs=-1
    ))
    r2_m, rmse_m = r2_scores.mean(), rmse_scores.mean()
    sweep_results.append({
        'name': name, 'rows': len(d), 'removed': removed,
        'skew': round(y.skew(), 2), 'r2': r2_m, 'rmse': rmse_m
    })
    print(f'{name:<22} {len(d):>8,} {removed:>9,} {y.skew():>7.2f} '
          f'{r2_m:>9.4f} {rmse_m:>9.1f}')

sr = pd.DataFrame(sweep_results)
best_strategy = sr.loc[sr['r2'].idxmax(), 'name']
print(f'\nBest strategy by R²: {best_strategy}')

In [ ]:
# ── Visualise the outlier sweep ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.arange(len(sr))
bar_cols = [C_TEAL if n == best_strategy else C_BLUE for n in sr['name']]

for ax, col, label in [
    (axes[0], 'r2',   'CV R²  (higher = better)'),
    (axes[1], 'rmse', 'CV RMSE — hours  (lower = better)'),
    (axes[2], 'rows', 'Rows retained'),
]:
    bars = ax.bar(x, sr[col], color=bar_cols, alpha=0.85, edgecolor='none')
    best_i = sr[col].idxmax() if col in ('r2','rows') else sr[col].idxmin()
    bars[best_i].set_edgecolor('#2C2C2A')
    bars[best_i].set_linewidth(1.8)
    for bar, val in zip(bars, sr[col]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + sr[col].max()*0.01,
                f'{val:.3f}' if col == 'r2' else f'{val:,.0f}',
                ha='center', va='bottom', fontsize=8, color=C_GRAY)
    ax.set_xticks(x)
    ax.set_xticklabels(sr['name'], rotation=30, ha='right', fontsize=8)
    ax.set_title(label, fontweight='500')

plt.suptitle('Outlier removal strategy comparison (RF, 5-fold CV)',
             fontweight='500', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Apply the best outlier strategy ──────────────────────────────────────
# Change this to any strategy name from the sweep above if you prefer a
# different trade-off between R² and rows retained.
CHOSEN_STRATEGY = best_strategy

df_clean = STRATEGIES[CHOSEN_STRATEGY].copy()
y_clean  = df_clean['hours_to_close']
X_clean  = df_clean.drop(columns=['hours_to_close'])
FEATURE_NAMES = list(X_clean.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Chosen strategy:  {CHOSEN_STRATEGY}')
print(f'Train rows: {len(X_train):,}   Test rows: {len(X_test):,}')
print(f'Target after cleaning — mean: {y_clean.mean():.1f}h  '
      f'median: {y_clean.median():.1f}h  skew: {y_clean.skew():.2f}')

# Scale features for neural network (tree models don't need this
# but we build both scaled and unscaled versions here)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Transform helpers ────────────────────────────────────────────────
def _apply_transform(y, transform):
    """Transform target before fitting."""
    if transform == 'log':
        return np.log1p(y)
    elif transform == 'sqrt':
        return np.sqrt(y)
    return y  # raw

def _invert_transform(y_pred, transform):
    """Back-transform predictions to original hours scale."""
    if transform == 'log':
        return np.expm1(np.clip(y_pred, 0, None))
    elif transform == 'sqrt':
        return np.clip(y_pred, 0, None) ** 2
    return y_pred

def evaluate(name, y_true, y_pred, n_features):
    """Compute R², Adjusted R², RMSE, MAE on the raw hours scale."""
    n  = len(y_true)
    r2 = r2_score(y_true, y_pred)
    # Adjusted R² penalises extra features: 1 - (1-R²)*(n-1)/(n-p-1)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    rmse   = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae    = float(mean_absolute_error(y_true, y_pred))
    return {
        'Model':        name,
        'Transform':    TARGET_TRANSFORM,
        'R²':           round(r2, 4),
        'Adj R²':       round(adj_r2, 4),
        'RMSE (h)':     round(rmse, 1),
        'MAE (h)':      round(mae, 1),
    }

n_features = X_train.shape[1]

print(f'Target transform: {TARGET_TRANSFORM}')
print(f'n_features for Adj R² denominator: {n_features}')
all_results = []

# ── ERT baselines ────────────────────────────────────────────────────
# We need the original ERT_hours column from before the leakage drop.
# Re-derive it from df_clean using the raw CSV's ERT column by
# merging on index — df_clean is a subset of the preprocessed df,
# which dropped ERT (Estimated Response Time). Instead we recompute
# ERT_hours from df_clean's ERT_days column (kept as a feature).
# ERT_days is in business days; multiply by 24 for calendar hours.
_ert_hours_all = df_clean['ERT_days'] * 24   # calendar hours
_ert_test      = _ert_hours_all.loc[X_test.index]   # test-set rows only
_y_test_arr    = np.array(y_test)

# Naïve baseline 1 — always predict the training mean
_mean_pred = np.full(len(y_test), float(y_train.mean()))
all_results.append(evaluate(
    'Baseline — train mean', y_test, _mean_pred, n_features=0
))

# Naïve baseline 2 — ERT as a direct prediction (raw days × 24h)
_ert_pred = np.array(_ert_test)
all_results.append(evaluate(
    'ERT raw (days × 24h)', y_test, _ert_pred, n_features=1
))

# Smart baseline — ERT group mean
# For each ERT bucket, use the mean actual close time from training data.
# This is the fairest domain-knowledge baseline: it asks 'how well does
# the ERT category predict close time if we calibrate from history?'
_ert_train = _ert_hours_all.loc[X_train.index]
_ert_means = (
    pd.Series(y_train.values, index=_ert_train.values)
    .groupby(level=0).mean()
    .to_dict()
)
_global_mean = float(y_train.mean())
_ert_group_pred = np.array([
    _ert_means.get(v, _global_mean) for v in _ert_test.values
])
all_results.append(evaluate(
    'ERT group mean (calibrated)', y_test, _ert_group_pred, n_features=1
))

print('Baselines computed:')
for r in all_results:
    print(f"  {r['Model']:<35}  R²={r['R²']:.4f}  "
          f"Adj R²={r['Adj R²']:.4f}  "
          f"RMSE={r['RMSE (h)']:.1f}h  MAE={r['MAE (h)']:.1f}h")

## Section 2 — Tree Models

In [ ]:
# ── Decision Tree, Random Forest, Gradient Boosting ──────────────────────
tree_models = {
    'Decision Tree': DecisionTreeRegressor(
        max_depth=10, min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=RANDOM_STATE
    ),
}

y_train_t = _apply_transform(y_train, TARGET_TRANSFORM)

for name, model in tree_models.items():
    model.fit(X_train, y_train_t)
    pred = _invert_transform(model.predict(X_test), TARGET_TRANSFORM)
    res  = evaluate(name, y_test, pred, n_features)
    all_results.append(res)
    print(f"{name:<22}  R²={res['R²']:.4f}  Adj R²={res['Adj R²']:.4f}  "
          f"RMSE={res['RMSE (h)']:.1f}h  MAE={res['MAE (h)']:.1f}h")

# XGBoost
if XGBOOST_OK:
    xgb = XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, eval_metric='rmse',
        n_jobs=-1, verbosity=0
    )
    xgb.fit(
        X_train, y_train_t,
        eval_set=[(X_test, _apply_transform(y_test, TARGET_TRANSFORM))],
        verbose=False
    )
    pred = _invert_transform(xgb.predict(X_test), TARGET_TRANSFORM)
    res  = evaluate('XGBoost', y_test, pred, n_features)
    all_results.append(res)
    print(f"{'XGBoost':<22}  R²={res['R²']:.4f}  Adj R²={res['Adj R²']:.4f}  "
          f"RMSE={res['RMSE (h)']:.1f}h  MAE={res['MAE (h)']:.1f}h")
else:
    print('XGBoost skipped — not installed.')

## Section 3 — Neural Network

Two approaches: sklearn's `MLPRegressor` (no extra install needed) and
a Keras deep network (requires TensorFlow).

In [ ]:
# ── sklearn MLP (always available) ────────────────────────────────────────
# Uses scaled features — tree models don't need scaling but MLP does.
print('Training sklearn MLP...')
mlp_sklearn = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=RANDOM_STATE,
)
mlp_sklearn.fit(X_train_scaled, y_train_t)
pred = _invert_transform(mlp_sklearn.predict(X_test_scaled), TARGET_TRANSFORM)
res  = evaluate('MLP (sklearn)', y_test, pred, n_features)
all_results.append(res)
print(f"{'MLP (sklearn)':<22}  R²={res['R²']:.4f}  Adj R²={res['Adj R²']:.4f}  "
      f"RMSE={res['RMSE (h)']:.1f}h  MAE={res['MAE (h)']:.1f}h")
print(f'  Stopped at iteration: {mlp_sklearn.n_iter_}')

In [ ]:
# ── Keras deep network (requires TensorFlow) ──────────────────────────────
if TF_OK:
    tf.random.set_seed(RANDOM_STATE)
    n_features = X_train_scaled.shape[1]

    keras_model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ])

    keras_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae'],
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=0
    )
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-6, verbose=0
    )

    print('Training Keras network...')
    history = keras_model.fit(
        X_train_scaled, y_train_t,
        validation_split=0.1,
        epochs=200,
        batch_size=256,
        callbacks=[early_stop, reduce_lr],
        verbose=0,
    )
    print(f'  Stopped at epoch: {len(history.history["loss"])}')

    keras_raw  = keras_model.predict(X_test_scaled, verbose=0).flatten()
    keras_pred = _invert_transform(keras_raw, TARGET_TRANSFORM)
    res = evaluate('Keras NN', y_test, keras_pred, n_features)
    all_results.append(res)
    print(f"{'Keras NN':<22}  R²={res['R²']:.4f}  Adj R²={res['Adj R²']:.4f}  "
          f"RMSE={res['RMSE (h)']:.1f}h  MAE={res['MAE (h)']:.1f}h")

    # Training curve
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(history.history['loss'],     color=C_BLUE,  lw=1.5, label='Train loss')
    ax.plot(history.history['val_loss'], color=C_RED,   lw=1.5, label='Val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE loss')
    ax.set_title('Keras training curve', fontweight='500')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Keras skipped — TensorFlow not installed.')
    print('Install with: pip install tensorflow')

In [ ]:
# ── Final comparison ──────────────────────────────────────────────────────
comp = pd.DataFrame(all_results).sort_values('R²', ascending=False)

print(f'=== All models — {CHOSEN_STRATEGY} | Transform: {TARGET_TRANSFORM} ===')
print(comp.to_string(index=False))
print(f'\nNote: Adj R² penalises model complexity (n={len(X_test):,}, p={n_features} features).')
print(f'Small gap between R² and Adj R² = model is not overfitting to feature count.')

MODEL_COLORS_MAP = {
    'Decision Tree':             C_GRAY,
    'Random Forest':             C_TEAL,
    'Gradient Boosting':         C_AMBER,
    'XGBoost':                   C_BLUE,
    'MLP (sklearn)':             '#7F77DD',
    'Keras NN':                  C_RED,
    'Baseline — train mean':     '#B4B2A9',  # light gray
    'ERT raw (days × 24h)':      '#F09595',  # light red
    'ERT group mean (calibrated)':'#85B7EB',  # light blue
}

# Split into ML models vs baselines for cleaner visual grouping
BASELINE_NAMES = [
    'Baseline — train mean',
    'ERT raw (days × 24h)',
    'ERT group mean (calibrated)',
]
comp_models    = comp[~comp['Model'].isin(BASELINE_NAMES)]
comp_baselines = comp[comp['Model'].isin(BASELINE_NAMES)]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
model_names = comp['Model'].tolist()
colors = [MODEL_COLORS_MAP.get(n, C_BLUE) for n in model_names]

for ax, col, label, higher_better in [
    (axes[0], 'R²',       'R²  (higher = better)',          True),
    (axes[1], 'Adj R²',   'Adj R²  (higher = better)',      True),
    (axes[2], 'RMSE (h)', 'RMSE — hours  (lower = better)', False),
    (axes[3], 'MAE (h)',  'MAE — hours  (lower = better)',  False),
]:
    vals = comp[col].tolist()
    bars = ax.bar(model_names, vals, color=colors, alpha=0.85, edgecolor='none')
    best_i = vals.index(max(vals) if higher_better else min(vals))
    bars[best_i].set_edgecolor('#2C2C2A')
    bars[best_i].set_linewidth(1.8)
    for bar, val in zip(bars, vals):
        offset = max(abs(v) for v in vals) * 0.02
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + offset,
                f'{val:.3f}' if col in ('R²','Adj R²') else f'{val:.0f}',
                ha='center', va='bottom', fontsize=7.5, color=C_GRAY)
    # Dashed line at ERT group mean as reference
    _ert_gm_val = comp[comp['Model']=='ERT group mean (calibrated)'][col].values
    if len(_ert_gm_val):
        ax.axhline(_ert_gm_val[0], color='#85B7EB',
                   linewidth=1, linestyle='--',
                   label='ERT group mean (domain baseline)')
        ax.legend(fontsize=7.5)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
    ax.set_title(f'{label}\n(best outlined)', fontweight='500')

plt.suptitle(
    f'Model comparison — {CHOSEN_STRATEGY} | Transform: {TARGET_TRANSFORM}\n'
    'Light gray = naïve baseline  |  Light red = ERT raw  |  '
    'Light blue = ERT calibrated  |  Dashed line = ERT group mean',
    fontweight='500', y=1.02, fontsize=9
)
plt.tight_layout()
plt.show()

# ── ERT breakdown table ───────────────────────────────────────────────
print('\n=== ERT accuracy breakdown (how well does each ERT value predict?) ===')
_ert_col = _ert_hours_all.rename('ERT_hours_col')
_df_breakdown = pd.concat([
    y_clean.rename('actual'),
    _ert_col,
], axis=1).dropna()
_ert_raw_labels = (
    df_clean.index.map(
        dict(zip(
            df_clean.index,
            (df_clean['ERT_days'].astype(str) + 'd').values
        ))
    )
)
_summary = _df_breakdown.groupby('ERT_hours_col').agg(
    n=('actual','count'),
    median_actual=('actual','median'),
    mean_actual=('actual','mean'),
    ert_hours=('ERT_hours_col','first'),
).reset_index(drop=True).sort_values('ert_hours')
_summary['ratio_med_vs_ERT'] = (_summary['median_actual'] / _summary['ert_hours']).round(2)
_summary['interpretation'] = _summary['ratio_med_vs_ERT'].apply(
    lambda r: 'accurate' if 0.7 <= r <= 1.3
    else (f'closes {1/r:.1f}× faster than ERT' if r < 0.7
    else f'closes {r:.1f}× slower than ERT')
)
print(_summary[['ert_hours','n','median_actual','mean_actual',
                 'ratio_med_vs_ERT','interpretation']]
      .rename(columns={'ert_hours':'ERT (h)','n':'n',
                       'median_actual':'Median actual (h)',
                       'mean_actual':'Mean actual (h)',
                       'ratio_med_vs_ERT':'Ratio',
                       'interpretation':'Interpretation'})
      .round(1).to_string(index=False))

print(f'\n% closed BEFORE ERT deadline: '
      f"{(_df_breakdown['actual'] < _df_breakdown['ERT_hours_col']).mean()*100:.1f}%")
print(f'% closed AFTER  ERT deadline: '
      f"{(_df_breakdown['actual'] > _df_breakdown['ERT_hours_col']).mean()*100:.1f}%")
print('\nKey: ERT is a conservative *promise*, not a prediction.')
print('Your models are learning the true resolution pattern — '
      'a different and harder problem.')

In [ ]:
# ── Predicted vs actual for all models ────────────────────────────────────
# All predictions are back-transformed to the original hours scale.
pred_dict = {}
pred_dict['Decision Tree']     = _invert_transform(
    tree_models['Decision Tree'].predict(X_test), TARGET_TRANSFORM)
pred_dict['Random Forest']     = _invert_transform(
    tree_models['Random Forest'].predict(X_test), TARGET_TRANSFORM)
pred_dict['Gradient Boosting'] = _invert_transform(
    tree_models['Gradient Boosting'].predict(X_test), TARGET_TRANSFORM)
if XGBOOST_OK:
    pred_dict['XGBoost'] = _invert_transform(
        xgb.predict(X_test), TARGET_TRANSFORM)
pred_dict['MLP (sklearn)'] = _invert_transform(
    mlp_sklearn.predict(X_test_scaled), TARGET_TRANSFORM)
if TF_OK:
    pred_dict['Keras NN'] = _invert_transform(
        keras_model.predict(X_test_scaled, verbose=0).flatten(), TARGET_TRANSFORM)

n_models = len(pred_dict)
fig, axes = plt.subplots(2, n_models, figsize=(5*n_models, 9))
if n_models == 1:
    axes = axes.reshape(2, 1)

y_test_arr = np.array(y_test)
step = max(1, len(y_test_arr) // 500)

for col, (name, y_pred) in enumerate(pred_dict.items()):
    color = MODEL_COLORS_MAP.get(name, C_BLUE)
    r2  = r2_score(y_test_arr, y_pred)
    mae = mean_absolute_error(y_test_arr, y_pred)

    # Predicted vs actual
    ax = axes[0, col]
    ax.scatter(y_test_arr[::step], y_pred[::step],
              alpha=0.2, s=6, color=color, edgecolors='none')
    lim = max(y_test_arr.max(), y_pred.max())
    ax.plot([0,lim],[0,lim], color=C_RED, lw=1.2, linestyle='--', label='Perfect')
    ax.set_xlabel('Actual hours')
    ax.set_ylabel('Predicted hours')
    ax.set_title(f'{name}\nR²={r2:.3f}', fontweight='500', fontsize=10)
    ax.legend(fontsize=8)

    # Residuals
    ax = axes[1, col]
    resid = y_test_arr - y_pred
    ax.scatter(y_pred[::step], resid[::step],
              alpha=0.2, s=6, color=color, edgecolors='none')
    ax.axhline(0, color=C_RED, lw=1.2, linestyle='--')
    ax.set_xlabel('Predicted hours')
    ax.set_ylabel('Residual')
    ax.set_title(f'Residuals — MAE={mae:.0f}h', fontsize=10)

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Predicted vs actual (top) | Residuals (bottom)',
             fontweight='500', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importances — tree models only ────────────────────────────────
TOP_N = 12
tree_imp_models = {'Random Forest': tree_models['Random Forest'],
                   'Gradient Boosting': tree_models['Gradient Boosting']}
if XGBOOST_OK:
    tree_imp_models['XGBoost'] = xgb

n = len(tree_imp_models)
fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
if n == 1: axes = [axes]

for ax, (name, model) in zip(axes, tree_imp_models.items()):
    imp = pd.Series(model.feature_importances_,
                   index=FEATURE_NAMES).sort_values(ascending=False).head(TOP_N)
    ax.barh(imp.index[::-1], imp.values[::-1],
           color=MODEL_COLORS_MAP.get(name, C_BLUE),
           alpha=0.85, height=0.7, edgecolor='none')
    ax.set_xlabel('Importance')
    ax.set_title(name, fontweight='500')
    ax.tick_params(axis='y', labelsize=8)
    for s in ['top','right']: ax.spines[s].set_visible(False)

plt.suptitle(f'Top {TOP_N} feature importances', fontweight='500', y=1.01)
plt.tight_layout()
plt.show()